# Chakra Ñawi · contador de broca en Colab (plan B / reentrenamiento con GPU)

Hace lo mismo que los scripts de `ml/` en la PC: genera datos **sintéticos**, entrena YOLOv8n de una clase, exporta ONNX (fp32 + int8) según `contracts/modelo-io.md` y evalúa el conteo.

1. Entorno de ejecución → Cambiar tipo → **GPU T4**.
2. Sube un zip de la carpeta `ml/` del repositorio (sin `.venv`, `datasets`, `runs`) como `ml.zip`, o clona el repositorio si es público.
3. Para reentrenar con **fotos reales** de gorgojos: pon los marcos rectificados 1216×1216 y sus etiquetas YOLO en `ml/datasets/real_v1/marcos/<split>/{images,labels}` y usa la celda opcional.

**Lo que entrena aquí con datos sintéticos NO es desempeño en broca real.**

In [ ]:
import os
if not os.path.exists('ml'):
    from google.colab import files
    up = files.upload()  # sube ml.zip
    !unzip -q ml.zip -d .
%cd ml
!pip -q install ultralytics onnx onnxruntime onnxslim opencv-python-headless
!nvidia-smi -L

In [ ]:
# Datos sintéticos (marcos 1216×1216 → mosaicos 640×640). En Colab hay 2 CPU: tarda ~20 min con 400/60/60.
!python sintetico.py --train 400 --val 60 --test 60 --workers 2

In [ ]:
# En GPU se puede entrenar más: cambia device a 0 y sube épocas.
from ultralytics import YOLO
m = YOLO('yolov8n.pt')
m.train(data='datasets/sint_v0/data.yaml', imgsz=640, epochs=60, batch=32, device=0, workers=2,
        project='runs', name='y8n_sint_v0', exist_ok=True, single_cls=True, cache='ram', plots=False,
        close_mosaic=5, mosaic=1.0, scale=0.15, translate=0.1, degrees=0.0, fliplr=0.5, flipud=0.5,
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, max_det=1000)

In [ ]:
# Exporta (opset 17, sin NMS, 640 fijo), crea int8, verifica E/S y escribe modelos/metadata.json
!python exportar.py --pesos runs/y8n_sint_v0/weights/best.pt --nombre broca-y8n-v0-sint --no-copiar
!python evaluar.py --nombre broca-y8n-v0-sint
!python contador_clasico.py --calibrar && python contador_clasico.py

In [ ]:
# Descarga los modelos y resultados; copia los .onnx a apps/campo/public/models/ en la PC
!zip -qr salida_ml.zip modelos resultados
from google.colab import files
files.download('salida_ml.zip')